In [2]:
import pandas as pd
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np

In [4]:
data = pd.read_csv("words.csv")

In [3]:
# Load BERT (or another transformer model)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Set model to evaluation mode
model.eval()

c:\Python 3.12\Projects\Code Projects\Python\Contexto\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prans\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling bac

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [5]:
def get_word_embedding(word):
    """Generate a word embedding using BERT."""
    with torch.no_grad():
        inputs = tokenizer(word, return_tensors="pt", padding=True, truncation=True)
        outputs = model(**inputs)
        # Extract token embeddings from the last hidden state
        last_hidden_state = outputs.last_hidden_state
        # Get the mean of the token embeddings
        word_embedding = last_hidden_state.mean(dim=1).squeeze().numpy()
    return word_embedding


In [6]:
# Apply the function to all words in the dataframe
vectorized_data = data.applymap(get_word_embedding)

# Convert to numpy arrays for efficient ML use
X = np.stack(vectorized_data.apply(lambda row: np.concatenate(row.values), axis=1))

print(X.shape)  # Shape: (num_samples, vector_size * num_columns)


C:\Users\prans\AppData\Local\Temp\ipykernel_1136\3763599977.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  vectorized_data = data.applymap(get_word_embedding)


(5, 8448)


In [7]:
print(X)

[[ 0.00598824 -0.21878974 -0.14484967 ... -0.0878674   0.01621566
  -0.14842574]
 [ 0.29594842  0.0032587  -0.0971468  ...  0.09031018 -0.7542868
  -0.13981795]
 [ 0.37083128 -0.06061547 -0.17603119 ...  0.02470805  0.00372593
  -0.29157403]
 [ 0.08292363 -0.17775063 -0.3022748  ...  0.00211898 -0.1823637
  -0.21499527]
 [ 0.33758235  0.27956733 -0.21267438 ...  0.1629174  -0.63462657
   0.16863844]]


In [32]:
with torch.no_grad():
    inputs = tokenizer("man", return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)
    # Extract token embeddings from the last hidden state
    last_hidden_state = outputs.last_hidden_state
    # Get the mean of the token embeddings
    woman_word_embedding = last_hidden_state.mean(dim=1).squeeze().numpy()


In [29]:
with torch.no_grad():
    inputs = tokenizer("woman", return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)
    # Extract token embeddings from the last hidden state
    last_hidden_state = outputs.last_hidden_state
    # Get the mean of the token embeddings
    crown_word_embedding = last_hidden_state.mean(dim=1).squeeze().numpy()

In [33]:
new_word_embedding = woman_word_embedding + crown_word_embedding

In [34]:
from sklearn.metrics.pairwise import cosine_similarity
# Get BERT's vocabulary embeddings
with torch.no_grad():
    vocab = tokenizer.get_vocab()
    index_to_token = {v: k for k, v in vocab.items()}  # Reverse vocab mapping
    word_vectors = model.embeddings.word_embeddings.weight.numpy()  # BERT token embeddings

# Compute cosine similarity
similarities = cosine_similarity([new_word_embedding], word_vectors)

# Find the closest word index
best_match_index = np.argmax(similarities)

# Get the decoded word
decoded_word = index_to_token[best_match_index]
print("Decoded word:", decoded_word)

Decoded word: emerging


In [26]:
len(similarities)

1

In [27]:
print(similarities)

[[-0.00934579 -0.00188119 -0.00725617 ...  0.01942693  0.05396982
   0.00809002]]
